# RAG preprocessing: *Lessons in Electric Circuits* (Volume I - DC)

This notebook turns the local HTML textbook mirror into clean, section-aware JSON chunks for a future Retrieval-Augmented Generation (RAG) application. The main principle is **preserve meaning before splitting text**: first keep headings, paragraphs, lists, and image positions together; only then split unusually long sections.

Run the cells in order. The short **Try it yourself** prompts are optional exercises that let you inspect each decision before moving on.

## 1. Imports and paths

`pathlib` finds the local files reliably on Windows, macOS, and Linux. Beautiful Soup parses the old HTML structure; `json` writes the final interchange formats. This notebook uses a simple word-based *token estimate* so no model-specific tokenizer is required yet. Before embedding with a specific model, replace `estimate_tokens` with that model's tokenizer.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from urllib.parse import urljoin
import hashlib
import json
import re

from bs4 import BeautifulSoup, Tag

DATA_DIR = Path("data/DC-html-vol-1")
OUTPUT_DIR = Path("data/processed")
VOLUME = "Volume I - DC"
SOURCE_BASE_URL = "https://ibiblio.org/kuphaldt/electricCircuits/DC/"

assert DATA_DIR.exists(), f"Could not find {DATA_DIR.resolve()}"

## 2. Load every local HTML file

The input is already downloaded, so this is deliberately a local loader rather than a web scraper. It includes the table-of-contents page in the inspection list, but later excludes it from the knowledge base because it is navigation rather than textbook content.

In [2]:
def load_html_files(data_dir: Path) -> list[dict]:
    """Load each HTML file once and keep its path for provenance."""
    pages = []
    for path in sorted(data_dir.glob("*.html")):
        # These legacy files are Latin-1 compatible; replacement keeps one bad byte
        # from stopping the entire preprocessing run.
        html = path.read_text(encoding="latin-1", errors="replace")
        pages.append({"path": path, "html": html})
    return pages

pages = load_html_files(DATA_DIR)
print(f"Loaded {len(pages)} HTML files")
print([page["path"].name for page in pages])

Loaded 20 HTML files
['DC_1.html', 'DC_10.html', 'DC_11.html', 'DC_12.html', 'DC_13.html', 'DC_14.html', 'DC_15.html', 'DC_16.html', 'DC_2.html', 'DC_3.html', 'DC_4.html', 'DC_5.html', 'DC_6.html', 'DC_7.html', 'DC_8.html', 'DC_9.html', 'DC_A1.html', 'DC_A2.html', 'DC_A3.html', 'index.html']


### Exercise 1 — inspect before parsing

A parser should follow the source, not assumptions. Run the next cell and compare the displayed tags with an HTML file. You should see three `h1` headings for a chapter (volume, chapter number, chapter title), `h2` section headings, paragraphs, lists, and images. Navigation is also present before the first real section.

In [3]:
def inspect_html(page: dict, limit: int = 12) -> None:
    soup = BeautifulSoup(page["html"], "html.parser")
    elements = soup.find_all(["h1", "h2", "h3", "p", "ul", "ol", "img"])
    print(page["path"].name)
    print(Counter(element.name for element in elements))
    for element in elements[:limit]:
        preview = element.get_text(" ", strip=True) or element.get("alt") or element.get("src") or "(empty)"
        print(f"<{element.name}> {preview[:100]}")

inspect_html(pages[0])

DC_1.html
Counter({'p': 157, 'img': 46, 'ul': 8, 'h2': 8, 'h1': 3})
<img> Previous
<img> Contents
<img> Next
<h1> Lessons In Electric Circuits -- Volume I
<h1> Chapter 1
<h1> BASIC CONCEPTS OF ELECTRICITY
<ul> Static electricity Conductors, insulators, and electron flow Electric circuits Voltage and current R
<h2> Static electricity
<p> It was discovered centuries ago that certain types of materials would mysteriously attract one anoth
<p> (empty)
<img> 00001.png
<p> Glass and silk aren't the only materials known to behave like this.  Anyone who has ever brushed up 


## 3. Small cleaning helpers

Cleaning is intentionally conservative. It collapses layout whitespace but does not strip units, punctuation, equations, or symbols. `is_navigation_image` removes the repeated Previous / Contents / Next controls without removing educational diagrams.

In [4]:
NAVIGATION_WORDS = {"previous", "contents", "next"}

def clean_text(value: str) -> str:
    """Keep readable text while removing HTML/layout-only whitespace."""
    return re.sub(r"\s+", " ", value).strip()

def is_navigation_image(tag: Tag) -> bool:
    alt = clean_text(tag.get("alt", "")).lower()
    filename = Path(tag.get("src", "")).name.lower()
    return alt in NAVIGATION_WORDS or filename in {"previous.jpg", "contents.jpg", "next.jpg"}

def heading_text(tag: Tag) -> str:
    return clean_text(tag.get_text(" ", strip=True))

def image_record(tag: Tag, source_url: str, html_path: Path) -> dict:
    src = tag.get("src", "")
    alt = clean_text(tag.get("alt", ""))
    label = alt or Path(src).stem or "diagram"
    return {
        "type": "image",
        "src": urljoin(source_url, src),
        "local_path": str(html_path.parent / src),
        "alt": alt or None,
        "description": None,
        "placeholder": f"[IMAGE: {label}]",
    }

## 4. Parse one page into semantic section documents

This is the central function. It starts after the first `h2`, so the mini table of contents and header navigation do not enter the corpus. Each `h2` begins a section and each `h3` begins a subsection. Paragraphs, lists, and images are kept in document order inside `content`; images also get a textual placeholder at their original position.

In [5]:
def page_identity(path: Path, soup: BeautifulSoup) -> tuple[str, str, str]:
    """Return volume, chapter/appendix label, and its title from the h1 headings."""
    h1s = [heading_text(tag) for tag in soup.find_all("h1")]
    label = next((text for text in h1s if text.lower().startswith(("chapter", "appendix"))), path.stem)
    title = h1s[-1] if h1s else path.stem
    return VOLUME, label, title

def parse_html_page(page: dict) -> list[dict]:
    """Create section-level documents from one textbook chapter or appendix."""
    path = page["path"]
    if path.name == "index.html":
        return []  # Loaded for inspection, then excluded because it is navigation.

    soup = BeautifulSoup(page["html"], "html.parser")
    volume, chapter, chapter_title = page_identity(path, soup)
    source_url = urljoin(SOURCE_BASE_URL, path.name)
    documents, content = [], []
    section, subsection = None, None

    def finish_section() -> None:
        nonlocal content
        if not section or not content:
            content = []
            return
        text = "\n\n".join(item["text"] for item in content)
        images = [item for item in content if item["type"] == "image"]
        documents.append({
            "volume": volume,
            "chapter": f"{chapter}: {chapter_title}",
            "section": section,
            "subsection": subsection,
            "content": content,
            "text": text,
            "images": images,
            "section_index": len(documents) + 1,
            "source_url": source_url,
            "source_file": path.name,
        })
        content = []

    # find_all preserves document order. Images inside a paragraph are handled by
    # that paragraph, so the separate image iteration skips them.
    for tag in soup.find_all(["h2", "h3", "p", "ul", "ol", "img"]):
        if tag.name == "h2":
            finish_section()
            section, subsection = heading_text(tag), None
        elif tag.name == "h3" and section:
            finish_section()
            subsection = heading_text(tag)
        elif not section:
            continue
        elif tag.name == "p":
            paragraph = clean_text(tag.get_text(" ", strip=True))
            if paragraph:
                content.append({"type": "paragraph", "text": paragraph})
            for image in tag.find_all("img"):
                if not is_navigation_image(image):
                    record = image_record(image, source_url, path)
                    content.append({**record, "text": record["placeholder"]})
        elif tag.name in {"ul", "ol"}:
            items = [clean_text(item.get_text(" ", strip=True)) for item in tag.find_all("li", recursive=False)]
            items = [item for item in items if item]
            if items:
                content.append({"type": "list", "text": "\n".join(f"- {item}" for item in items)})
        elif tag.name == "img" and tag.parent and tag.parent.name != "p" and not is_navigation_image(tag):
            record = image_record(tag, source_url, path)
            content.append({**record, "text": record["placeholder"]})

    finish_section()
    return documents

### Exercise 2 — verify the intermediate representation

Run this cell and inspect `section_documents[0]`. Notice that it still contains `content` (the detailed ordered blocks) as well as `text` (a convenient version for chunking). Confirm that a diagram is represented by both image metadata and an `[IMAGE: ...]` placeholder.

In [6]:
section_documents = [document for page in pages for document in parse_html_page(page)]
print(f"Built {len(section_documents)} section-level documents from {len(pages) - 1} content pages.")
print(json.dumps(section_documents[0], indent=2, ensure_ascii=False)[:2500])

Built 162 section-level documents from 19 content pages.
{
  "volume": "Volume I - DC",
  "chapter": "Chapter 1: BASIC CONCEPTS OF ELECTRICITY",
  "section": "Static electricity",
  "subsection": null,
  "content": [
    {
      "type": "paragraph",
      "text": "It was discovered centuries ago that certain types of materials would mysteriously attract one another after being rubbed together. For example: after rubbing a piece of silk against a piece of glass, the silk and glass would tend to stick together. Indeed, there was an attractive force that could be demonstrated even when the two materials were separated:"
    },
    {
      "type": "image",
      "src": "https://ibiblio.org/kuphaldt/electricCircuits/DC/00001.png",
      "local_path": "data\\DC-html-vol-1\\00001.png",
      "alt": null,
      "description": null,
      "placeholder": "[IMAGE: 00001]",
      "text": "[IMAGE: 00001]"
    },
    {
      "type": "paragraph",
      "text": "Glass and silk aren't the only material

## 5. Chunk only long semantic sections

A small section stays whole. A long one is divided on paragraph/list/image-block boundaries and receives a small overlap from the previous chunk. The heading context is repeated in every final chunk so retrieved text remains understandable by itself.

The word count below is an approximation, not the exact token count of a future embedding model. It is sufficient for this initial 400–700-token target; use the embedding model's tokenizer before enforcing a production limit.
MAX_TOKENS actually controls chunk size.

In [7]:
TARGET_TOKENS = 550
MAX_TOKENS = 700
OVERLAP_TOKENS = 75

def estimate_tokens(text: str) -> int:
    """A transparent approximate token count: punctuation/words are counted as units."""
    return len(re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE))

def context_header(document: dict) -> str:
    headings = [document["volume"], document["chapter"], f"Section: {document['section']}"]
    if document["subsection"]:
        headings.append(f"Subsection: {document['subsection']}")
    return "\n".join(headings)

def overlap_blocks(blocks: list[dict], limit: int) -> list[dict]:
    selected, size = [], 0
    for block in reversed(blocks):
        block_size = estimate_tokens(block["text"])
        if size + block_size > limit:
            break
        selected.insert(0, block)
        size += block_size
    return selected

def split_long_block(block: dict, limit: int) -> list[dict]:
    """Split only an overlong paragraph; prefer sentences, then words as a last resort."""
    if block["type"] == "image" or estimate_tokens(block["text"]) <= limit:
        return [block]
    sentences = re.split(r"(?<=[.!?])\s+", block["text"])
    pieces, current = [], ""
    for sentence in sentences:
        candidate = f"{current} {sentence}".strip()
        if current and estimate_tokens(candidate) > limit:
            pieces.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        pieces.append(current)
    # A single unusually long sentence is split by words only when necessary.
    final_pieces = []
    for piece in pieces:
        current_words = []
        for word in piece.split():
            candidate = " ".join(current_words + [word])
            if current_words and estimate_tokens(candidate) > limit:
                final_pieces.append(" ".join(current_words))
                current_words = [word]
            else:
                current_words.append(word)
        if current_words:
            final_pieces.append(" ".join(current_words))
    return [{**block, "text": piece} for piece in final_pieces]

def chunk_document(document: dict) -> list[dict]:
    """Keep a section intact unless its content exceeds MAX_TOKENS."""
    header = context_header(document)
    groups, current = [], []
    content_limit = MAX_TOKENS - estimate_tokens(header) - 5
    blocks = [piece for block in document["content"] for piece in split_long_block(block, content_limit)]
    for block in blocks:
        proposed = current + [block]
        if current and estimate_tokens(header + "\n\n" + "\n\n".join(item["text"] for item in proposed)) > MAX_TOKENS:
            groups.append(current)
            available_overlap = max(0, content_limit - estimate_tokens(block["text"]))
            current = overlap_blocks(current, min(OVERLAP_TOKENS, available_overlap)) + [block]
        else:
            current = proposed
    if current:
        groups.append(current)

    page_code = document["source_file"].replace(".html", "").lower().replace("_", "")
    chunks = []
    for number, blocks in enumerate(groups, start=1):
        text = header + "\n\n" + "\n\n".join(block["text"] for block in blocks)
        chunks.append({
            "chunk_id": f"{page_code}_sec{document['section_index']:02d}_{number:03d}",
            "text": text,
            "volume": document["volume"],
            "chapter": document["chapter"],
            "section": document["section"],
            "subsection": document["subsection"],
            "source_url": document["source_url"],
            "source_file": document["source_file"],
            "images": [block for block in blocks if block["type"] == "image"],
            "token_count": estimate_tokens(text),
        })
    return chunks

chunks = [chunk for document in section_documents for chunk in chunk_document(document)]
print(f"Created {len(chunks)} chunks.")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:2000])

Created 335 chunks.
{
  "chunk_id": "dc1_sec01_001",
  "text": "Volume I - DC\nChapter 1: BASIC CONCEPTS OF ELECTRICITY\nSection: Static electricity\n\nIt was discovered centuries ago that certain types of materials would mysteriously attract one another after being rubbed together. For example: after rubbing a piece of silk against a piece of glass, the silk and glass would tend to stick together. Indeed, there was an attractive force that could be demonstrated even when the two materials were separated:\n\n[IMAGE: 00001]\n\nGlass and silk aren't the only materials known to behave like this. Anyone who has ever brushed up against a latex balloon only to find that it tries to stick to them has experienced this same phenomenon. Paraffin wax and wool cloth are another pair of materials early experimenters recognized as manifesting attractive forces after being rubbed together:\n\n[IMAGE: 00002]\n\nThis phenomenon became even more interesting when it was discovered that identical material

## 6. Validate before export

Validation makes common failures visible: empty chunks, unexpectedly tiny chunks, navigation-only chunks, and duplicate text. Extremely short chunks are reported for review rather than automatically deleted because a concise formula or figure caption can still be useful educational content.

In [8]:
counts = [chunk["token_count"] for chunk in chunks]
fingerprints = Counter(hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest() for chunk in chunks)
empty = [chunk["chunk_id"] for chunk in chunks if not chunk["text"].strip()]
very_short = [chunk["chunk_id"] for chunk in chunks if chunk["token_count"] < 40]
navigation_only = [chunk["chunk_id"] for chunk in chunks if set(clean_text(chunk["text"]).lower().split()) <= NAVIGATION_WORDS]
duplicates = [fingerprint for fingerprint, count in fingerprints.items() if count > 1]

print({
    "sections": len(section_documents),
    "chunks": len(chunks),
    "min_tokens": min(counts),
    "max_tokens": max(counts),
    "average_tokens": round(sum(counts) / len(counts), 1),
    "chunks_with_images": sum(bool(chunk["images"]) for chunk in chunks),
    "empty_chunks": empty,
    "very_short_chunks": very_short[:10],
    "navigation_only_chunks": navigation_only,
    "duplicate_text_groups": len(duplicates),
})

for example in chunks[:3]:
    print(f"\n{example['chunk_id']} | {example['token_count']} estimated tokens")
    print(example["text"][:500])

{'sections': 162, 'chunks': 335, 'min_tokens': 41, 'max_tokens': 700, 'average_tokens': 499.3, 'chunks_with_images': 232, 'empty_chunks': [], 'very_short_chunks': [], 'navigation_only_chunks': [], 'duplicate_text_groups': 0}

dc1_sec01_001 | 649 estimated tokens
Volume I - DC
Chapter 1: BASIC CONCEPTS OF ELECTRICITY
Section: Static electricity

It was discovered centuries ago that certain types of materials would mysteriously attract one another after being rubbed together. For example: after rubbing a piece of silk against a piece of glass, the silk and glass would tend to stick together. Indeed, there was an attractive force that could be demonstrated even when the two materials were separated:

[IMAGE: 00001]

Glass and silk aren't the only materials

dc1_sec01_002 | 642 estimated tokens
Volume I - DC
Chapter 1: BASIC CONCEPTS OF ELECTRICITY
Section: Static electricity

Postulating the existence of a single "fluid" that was either gained or lost through rubbing accounted best for th

### Exercise 3 — quality check

Open one example that contains an image and one from a long chapter. Ask: does the heading context explain the text? Is the image placeholder near its surrounding explanation? If a chunk is too large for the embedding model you later choose, lower `MAX_TOKENS` and rerun the chunking and validation cells.

In [25]:
long_found = False
with_image_found = False

token_size = MAX_TOKENS - 10
for example in chunks:

    if not long_found and example["token_count"] > token_size :
        long_found = True
        print(f"\n BIG CHUNK EXAMPLE:\n {example['chunk_id']} | {example['token_count']} estimated tokens")
        print(example["text"][:500])

    if not with_image_found:
        if example["images"] :

            print(f"\n IMAGE EXAMPLE:\n {example['images'][0]}")
            with_image_found = True

    if long_found and with_image_found:
        break


 IMAGE EXAMPLE:
 {'type': 'image', 'src': 'https://ibiblio.org/kuphaldt/electricCircuits/DC/00001.png', 'local_path': 'data\\DC-html-vol-1\\00001.png', 'alt': None, 'description': None, 'placeholder': '[IMAGE: 00001]', 'text': '[IMAGE: 00001]'}

 BIG CHUNK EXAMPLE:
 dc1_sec04_003 | 693 estimated tokens
Volume I - DC
Chapter 1: BASIC CONCEPTS OF ELECTRICITY
Section: Voltage and current

Because voltage is an expression of potential energy, representing the possibility or potential for energy release as the electrons move from one "level" to another, it is always referenced between two points. Consider the water reservoir analogy:

[IMAGE: 00020]

Because of the difference in the height of the drop, there's potential for much more energy to be released from the reservoir through the piping to lo


## 7. Export JSON and JSONL

`chunks.json` is convenient to inspect as one list. `chunks.jsonl` stores one chunk per line, which is convenient for streaming tools and later ingestion. Both files contain the same final records.

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_path = OUTPUT_DIR / "chunks.json"
jsonl_path = OUTPUT_DIR / "chunks.jsonl"

json_path.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
with jsonl_path.open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Wrote {json_path} and {jsonl_path}")
print(f"JSONL records: {sum(1 for _ in jsonl_path.open(encoding='utf-8'))}")

Wrote data\processed\chunks.json and data\processed\chunks.jsonl
JSONL records: 335
